### **Vigenère Cipher and Its Cryptanalysis**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/Vigenere_Cipher.ipynb)

> The **Vigenère cipher** shifts each letter by an amount given by a repeating keyword, so the same plaintext letter becomes different ciphertext letters. It resisted attacks for three centuries, but the **repetition of the key** gives it away. This notebook encrypts a text, finds the key length with the **index of coincidence**, recovers the key with frequency analysis, and confirms the result with a simple **English detector**.

> [!WARNING]
> Educational code. Never use these implementations to protect real data: use a vetted library (for example, `cryptography`) with modern, authenticated algorithms.

In [1]:
# Encryption and decryption: only letters are shifted, and the key advances only on letters
import string

ALPHABET = string.ascii_uppercase

def vigenere(text, key, direction=1):
    shifts = [ALPHABET.index(k) for k in key.upper()]
    result, i = [], 0
    for char in text.upper():
        if char in ALPHABET:
            result.append(ALPHABET[(ALPHABET.index(char) + direction * shifts[i % len(shifts)]) % 26])
            i += 1
        else:
            result.append(char)
    return "".join(result)

PLAINTEXT = """Cryptography has always been a contest between the people who build ciphers and the people who break them. For centuries the strongest systems were polyalphabetic ciphers, which change the substitution for every letter of the message according to a secret keyword. They hide the simple letter frequencies that make a single substitution easy to attack, and for that reason they were once called unbreakable. In the nineteenth century, however, careful analysts noticed that the keyword repeats, and that a repeating keyword leaves its own pattern in the ciphertext. Once the length of the keyword is known, the message splits into several simple shift ciphers, and each of them falls to ordinary frequency analysis. The lesson still matters today: any structure that repeats inside an encryption scheme is an opening for an attacker, and modern designs are built to remove it."""
KEY = "ANIMAL"
ciphertext = vigenere(PLAINTEXT, KEY)
print(ciphertext[:150], "...")
assert vigenere(ciphertext, KEY, -1) == PLAINTEXT.upper()

CEGBTZGEIBHJ HNA MLHALA NEPN N KANEEFB NEEWRMZ TSE CMAPWE JPA BFIYL OIAHRZE AYD GPQ PPOCTQ WSO OZQAV TUMY. FZR PMZTFRVME TSE FBDOYGRAF SJSGMYS HEEM BO ...


#### **01. Finding the Key Length: Index of Coincidence**

The **index of coincidence (IoC)** is the probability that two random letters of a text are equal: about **0.066** for English and **0.038** for uniformly random letters. If the text is split into columns using the right key length, every column is a simple Caesar cipher and keeps the English IoC.

In [2]:
# Average IoC of the columns for each candidate key length
letters = "".join(c for c in ciphertext if c in ALPHABET)

def index_of_coincidence(text):
    n = len(text)
    return sum(text.count(c) * (text.count(c) - 1) for c in ALPHABET) / (n * (n - 1))

print(f"{'length':>6} | {'average IoC':>11}")
scores = {}
for length in range(1, 13):
    scores[length] = sum(index_of_coincidence(letters[i::length]) for i in range(length)) / length
    print(f"{length:>6} | {scores[length]:>11.4f}")
key_length = next(length for length, ioc in scores.items() if ioc > 0.058)
print(f"\nSmallest length with an English-like IoC: {key_length}")

length | average IoC
     1 |      0.0434
     2 |      0.0504
     3 |      0.0515
     4 |      0.0499
     5 |      0.0431
     6 |      0.0701
     7 |      0.0420
     8 |      0.0507
     9 |      0.0505
    10 |      0.0509
    11 |      0.0428
    12 |      0.0687

Smallest length with an English-like IoC: 6


#### **02. Recovering Each Letter of the Key**

Each column is a Caesar cipher, broken by choosing the shift whose letter counts best match English (chi-squared).

In [3]:
# Chi-squared frequency analysis on each column
ENGLISH_FREQ = [8.2, 1.5, 2.8, 4.3, 12.7, 2.2, 2.0, 6.1, 7.0, 0.15, 0.8, 4.0, 2.4,
                6.7, 7.5, 1.9, 0.1, 6.0, 6.3, 9.1, 2.8, 1.0, 2.4, 0.15, 2.0, 0.07]

def best_shift(column):
    def chi(shift):
        counts = [0] * 26
        for c in column:
            counts[(ALPHABET.index(c) - shift) % 26] += 1
        return sum((counts[i] - len(column) * f / 100) ** 2 / (len(column) * f / 100) for i, f in enumerate(ENGLISH_FREQ))
    return min(range(26), key=chi)

recovered_key = "".join(ALPHABET[best_shift(letters[i::key_length])] for i in range(key_length))
print("Recovered key:", recovered_key)
assert recovered_key == KEY

Recovered key: ANIMAL


#### **03. Confirming with an English Detector**

An automated attack needs to recognize success. A simple detector counts how many words belong to a list of common English words.

In [4]:
# Share of common English words: high for the right key, low for a wrong one
COMMON_WORDS = set("""THE OF AND TO A IN IS IT THAT FOR WAS ON ARE AS WITH THEY BE AT ONE HAVE THIS FROM BY
OR HAD NOT BUT WHAT ALL WERE WHEN WE THERE CAN AN YOUR WHICH THEIR IF DO WILL EACH ABOUT HOW UP OUT THEM
THEN SHE MANY SOME SO THESE WOULD OTHER INTO HAS MORE TWO TIME LIKE NO JUST HIM KNOW ONCE STILL BUILT""".split())

def english_score(text):
    words = [w.strip(string.punctuation) for w in text.upper().split()]
    return sum(w in COMMON_WORDS for w in words) / max(len(words), 1)

for key in (recovered_key, "ANIMAX", "BANANA"):
    print(f"key={key}: {english_score(vigenere(ciphertext, key, -1)):.0%} common English words")
print("\n" + vigenere(ciphertext, recovered_key, -1)[:200], "...")

key=ANIMAL: 44% common English words
key=ANIMAX: 21% common English words
key=BANANA: 0% common English words

CRYPTOGRAPHY HAS ALWAYS BEEN A CONTEST BETWEEN THE PEOPLE WHO BUILD CIPHERS AND THE PEOPLE WHO BREAK THEM. FOR CENTURIES THE STRONGEST SYSTEMS WERE POLYALPHABETIC CIPHERS, WHICH CHANGE THE SUBSTITUTIO ...


#### **Key Takeaways**

* A repeating key leaves a measurable pattern: the index of coincidence reveals its length.
* With the length known, the cipher splits into simple shifts broken by frequency analysis.
* The only way to remove the pattern is a key as long as the message and never reused: the one-time pad.